第 1 段：安装依赖库

In [ ]:
# 安装所需库（在 Jupyter 中运行一次即可）
!pip install scikit-learn nltk

第 2 段：文本预处理函数

In [1]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# 首次运行需要下载停用词
nltk.download('stopwords')

def preprocess_text(text):
    # 转换为小写
    text = text.lower()
    # 移除特殊字符和数字（只保留字母和空格）
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # 分词
    words = text.split()
    # 移除停用词
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words]
    # 词干提取
    stemmer = PorterStemmer()
    words = [stemmer.stem(word) for word in words]
    return ' '.join(words)

# 测试
sample = "This is a sample text for preprocessing!"
print(preprocess_text(sample))
# 输出: 'sampl text preprocess'

C:\Users\Administrator\.conda\envs\rl\lib\ssl.py:570: UserWarning: unable to load Windows certificates, some may be corrupted
  warnings.warn("unable to load Windows certificates, some may be corrupted")


sampl text preprocess


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


第 3 段：数据准备（加载 20 Newsgroups 数据集）

In [2]:
from sklearn.datasets import fetch_20newsgroups

# 选择4个类别作为示例
categories = ['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med']

# 加载训练集和测试集
newsgroups_train = fetch_20newsgroups(subset='train', categories=categories)
newsgroups_test = fetch_20newsgroups(subset='test', categories=categories)

print(f"训练集样本数: {len(newsgroups_train.data)}")
print(f"测试集样本数: {len(newsgroups_test.data)}")
print(f"类别名称: {newsgroups_train.target_names}")

训练集样本数: 2257
测试集样本数: 1502
类别名称: ['alt.atheism', 'comp.graphics', 'sci.med', 'soc.religion.christian']


第 4 段：特征提取（TF-IDF）

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 创建 TF-IDF 向量化器，限制最大特征数为 5000
vectorizer = TfidfVectorizer(max_features=5000)

# 转换训练集和测试集
X_train = vectorizer.fit_transform(newsgroups_train.data)
X_test = vectorizer.transform(newsgroups_test.data)

# 标签
y_train = newsgroups_train.target
y_test = newsgroups_test.target

print(f"训练集特征矩阵形状: {X_train.shape}")
print(f"测试集特征矩阵形状: {X_test.shape}")

训练集特征矩阵形状: (2257, 5000)
测试集特征矩阵形状: (1502, 5000)


第 5 段：模型训练（逻辑回归）

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 创建并训练模型
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 预测测试集
y_pred = model.predict(X_test)

# 评估模型
print(f"准确率: {accuracy_score(y_test, y_pred):.2f}")
print("\n分类报告:")
print(classification_report(y_test, y_pred, target_names=newsgroups_test.target_names))

准确率: 0.90

分类报告:
                        precision    recall  f1-score   support

           alt.atheism       0.94      0.78      0.85       319
         comp.graphics       0.87      0.96      0.92       389
               sci.med       0.94      0.89      0.91       396
soc.religion.christian       0.87      0.95      0.91       398

              accuracy                           0.90      1502
             macro avg       0.91      0.90      0.90      1502
          weighted avg       0.91      0.90      0.90      1502



补充 1：朴素贝叶斯分类器

In [5]:
from sklearn.naive_bayes import MultinomialNB

# 训练朴素贝叶斯模型
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# 预测与评估
y_pred_nb = nb_model.predict(X_test)
print("朴素贝叶斯准确率:", accuracy_score(y_test, y_pred_nb))

朴素贝叶斯准确率: 0.8768308921438083


补充 2：支持向量机（SVM）

In [6]:
from sklearn.svm import SVC

# 训练 SVM 模型（使用线性核，适合高维稀疏数据）
svm_model = SVC(kernel='linear')
svm_model.fit(X_train, y_train)

# 预测与评估
y_pred_svm = svm_model.predict(X_test)
print("SVM 准确率:", accuracy_score(y_test, y_pred_svm))

SVM 准确率: 0.9127829560585885


补充 3：使用 N-gram 特征

In [7]:
# 使用 1-gram 和 2-gram 组合
vectorizer_ngram = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_ngram = vectorizer_ngram.fit_transform(newsgroups_train.data)
X_test_ngram = vectorizer_ngram.transform(newsgroups_test.data)

# 训练逻辑回归
model_ngram = LogisticRegression(max_iter=1000)
model_ngram.fit(X_train_ngram, y_train)
y_pred_ngram = model_ngram.predict(X_test_ngram)

print(f"使用 N-gram 的准确率: {accuracy_score(y_test, y_pred_ngram):.2f}")

使用 N-gram 的准确率: 0.90


补充 4：处理类别不平衡（使用 class_weight）

In [8]:
# 使用类别权重处理不平衡数据
model_weighted = LogisticRegression(max_iter=1000, class_weight='balanced')
model_weighted.fit(X_train, y_train)
y_pred_weighted = model_weighted.predict(X_test)

print(f"使用类别权重的准确率: {accuracy_score(y_test, y_pred_weighted):.2f}")

使用类别权重的准确率: 0.90


补充 5：完整的文本分类流水线（含预处理）

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# 构建 Pipeline
pipeline = Pipeline([
    ('preprocess', None),  # 可以自定义预处理函数，但 TfidfVectorizer 自带预处理
    ('tfidf', TfidfVectorizer(max_features=5000, lowercase=True, stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000))
])

# 使用原始数据（不经过自定义预处理，因为 TfidfVectorizer 已包含 lowercase 和 stop_words）
X = newsgroups_train.data
y = newsgroups_train.target

# 分割训练集和验证集
X_train_pipe, X_val_pipe, y_train_pipe, y_val_pipe = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 训练
pipeline.fit(X_train_pipe, y_train_pipe)

# 评估
y_pred_pipe = pipeline.predict(X_val_pipe)
print(f"Pipeline 准确率: {accuracy_score(y_val_pipe, y_pred_pipe):.2f}")

Pipeline 准确率: 0.95
